# AI Nutrition Intelligence System
## Random Forest Baseline Model
**Milestone 2 — Week 3 — Day 15**

---

### 1. Project Objective

"We are building a machine learning model that predicts nutritional deficiency-related disease categories from the available nutritional, laboratory, symptom and demographic features."

> ⚠️ **Important Clinical Notice**: This system is designed as a **prediction and risk-support tool** to assist healthcare professionals and individuals in identifying early deficiency risks. It is **NOT** a standalone medical diagnosis system.

#### 🎯 Target Encodings & Labels
- **`0`**: **Anemia**
- **`1`**: **Healthy**
- **`2`**: **Night_Blindness**
- **`3`**: **Rickets_Osteomalacia**
- **`4`**: **Scurvy**



### 2. Import Libraries

We import only the essential Python data science and machine learning libraries.



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
import joblib
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Required libraries imported successfully!")


✅ Required libraries imported successfully!



#### 📖 Library Explanations:
- **Pandas (`pd`)**: Data structure manipulation and DataFrame operations.
- **NumPy (`np`)**: High-performance mathematical computations and array handling.
- **Matplotlib (`plt`) & Seaborn (`sns`)**: Data visualization and confusion matrix plotting.
- **Scikit-Learn (`sklearn`)**: ML algorithms (`RandomForestClassifier`), metrics evaluation, and data partitioning.
- **Joblib**: Saving and loading trained machine learning models and preprocessor pipelines.



### 3. Load Dataset

Loading the verified baseline dataset (`datasets/deficiency/deficiency_cleaned.csv`).



In [2]:
dataset_path = 'datasets/deficiency/deficiency_cleaned.csv'
df = pd.read_csv(dataset_path)

print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

print("--- First 5 Rows ---")
print(df.head(5))

print("\n--- Column Names ---")
print(list(df.columns))

print("\n--- Data Types Summary ---")
print(df.dtypes.value_counts())


=== DATASET OVERVIEW ===
Shape: 4000 rows x 49 columns

--- First 5 Rows ---
   age   bmi  ...  latitude_region_Mid  disease_diagnosis
0   79  24.8  ...                    1                  1
1   77  39.9  ...                    0                  3
2   24  26.4  ...                    0                  1
3   69  23.1  ...                    0                  0
4   63  29.6  ...                    0                  1

[5 rows x 49 columns]

--- Column Names ---
['age', 'bmi', 'vitamin_a_percent_rda', 'vitamin_c_percent_rda', 'vitamin_d_percent_rda', 'vitamin_e_percent_rda', 'vitamin_b12_percent_rda', 'folate_percent_rda', 'calcium_percent_rda', 'iron_percent_rda', 'hemoglobin_g_dl', 'serum_vitamin_d_ng_ml', 'serum_vitamin_b12_pg_ml', 'serum_folate_ng_ml', 'symptoms_count', 'has_night_blindness', 'has_fatigue', 'has_bleeding_gums', 'has_bone_pain', 'has_muscle_weakness', 'has_numbness_tingling', 'has_memory_problems', 'has_pale_skin', 'gender_Female', 'gender_Male', 'smoking_status_

### 4. Separate Features ($X$) and Target ($y$)

- **$X$**: Input features used by the model to learn patterns.
- **$y$**: Target outcome (`disease_diagnosis`) that the model predicts.



In [3]:
X = df.drop(columns=["disease_diagnosis"])
y = df["disease_diagnosis"]

print("=== SEPARATION SUMMARY ===")
print(f"Input Feature Matrix (X) Shape : {X.shape[0]} rows x {X.shape[1]} columns")
print(f"Target Label Vector (y) Shape   : {y.shape[0]} rows")


=== SEPARATION SUMMARY ===
Input Feature Matrix (X) Shape : 4000 rows x 48 columns
Target Label Vector (y) Shape   : 4000 rows



### 5. Train / Validation / Test Split

We partition data into 70% Training, 15% Validation, and 15% Test using `random_state=42` and `stratify=y`.



In [4]:
# 70% Train, 30% Temp (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Split Temp into 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("=== SPLIT SHAPES ===")
print(f"Training Set   (70%) : {X_train.shape[0]} samples")
print(f"Validation Set (15%) : {X_val.shape[0]} samples")
print(f"Test Set       (15%) : {X_test.shape[0]} samples")

print("\n=== TARGET CLASS DISTRIBUTION ACROSS SPLITS ===")
dist_df = pd.DataFrame({
    'Disease Class': ['Anemia (0)', 'Healthy (1)', 'Night_Blindness (2)', 'Rickets_Osteomalacia (3)', 'Scurvy (4)'],
    'Train Count': y_train.value_counts().sort_index().values,
    'Train (%)': (y_train.value_counts(normalize=True).sort_index() * 100).values.round(2),
    'Val Count': y_val.value_counts().sort_index().values,
    'Val (%)': (y_val.value_counts(normalize=True).sort_index() * 100).values.round(2),
    'Test Count': y_test.value_counts().sort_index().values,
    'Test (%)': (y_test.value_counts(normalize=True).sort_index() * 100).values.round(2)
})
print(dist_df.to_string(index=False))


NameError: name 'X' is not defined

#### 📖 Why Stratification is Important:
Target classes are severely imbalanced. **Night Blindness (3.05%)** and **Scurvy (2.38%)** are rare minority classes. Stratified splitting enforces that every split contains the exact same percentage of each disease class, preventing evaluation bias.



### 6. Load Existing Preprocessor

We load `backend/ml/artifacts/preprocessor.joblib` and transform all dataset partitions.

> ⚠️ **Data Leakage Notice**: We apply ONLY `.transform()`. `.fit()` must NEVER be called on validation or test sets.



In [5]:
preprocessor_path = 'backend/ml/artifacts/preprocessor.joblib'
preprocessor = joblib.load(preprocessor_path)

# Transform datasets
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("=== TRANSFORMED MATRIX SHAPES ===")
print(f"X_train_processed : {X_train_processed.shape}")
print(f"X_val_processed   : {X_val_processed.shape}")
print(f"X_test_processed  : {X_test_processed.shape}")


NameError: name 'X_train' is not defined

#### 📖 What is Data Leakage?
Data leakage happens when information from validation or test sets influences training. If scalers are fitted on the whole dataset, test set means and variances bleed into training, giving misleadingly high performance scores.



### 7. Train Random Forest Baseline

We train `RandomForestClassifier` with `class_weight="balanced"`.



In [6]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Train strictly using training data
model.fit(X_train_processed, y_train)

print("✅ Random Forest Baseline trained successfully!")


NameError: name 'X_train_processed' is not defined

#### 📖 Why Use `class_weight="balanced"`?
`class_weight="balanced"` automatically penalizes classification errors on minority classes (Night Blindness & Scurvy) more heavily, forcing the decision trees to focus on rare disease signals.



### 8. Validation Predictions


In [7]:
y_val_pred = model.predict(X_val_processed)
y_val_proba = model.predict_proba(X_val_processed)

print(f"Validation Predictions Generated : {len(y_val_pred)} predictions")


NameError: name 'model' is not defined

### 9. Validation Evaluation


In [8]:
val_acc = accuracy_score(y_val, y_val_pred)
val_prec_w = precision_score(y_val, y_val_pred, average="weighted")
val_rec_w = recall_score(y_val, y_val_pred, average="weighted")
val_f1_w = f1_score(y_val, y_val_pred, average="weighted")
val_f1_macro = f1_score(y_val, y_val_pred, average="macro")

print("=== VALIDATION METRICS ===")
print(f"Accuracy           : {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"Weighted Precision : {val_prec_w:.4f}")
print(f"Weighted Recall    : {val_rec_w:.4f}")
print(f"Weighted F1-Score  : {val_f1_w:.4f}")
print(f"Macro F1-Score     : {val_f1_macro:.4f}")

target_names = ['Anemia', 'Healthy', 'Night_Blindness', 'Rickets_Osteomalacia', 'Scurvy']
print("\n--- Classification Report (Validation) ---")
print(classification_report(y_val, y_val_pred, target_names=target_names))

cm_val = confusion_matrix(y_val, y_val_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Validation Confusion Matrix - Random Forest Baseline', fontweight='bold', fontsize=12)
plt.xlabel('Predicted Diagnosis')
plt.ylabel('Actual Diagnosis')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


NameError: name 'y_val' is not defined

### 10. Test Evaluation

We evaluate our final baseline model on the held-out Test Set.



In [9]:
y_test_pred = model.predict(X_test_processed)
y_test_proba = model.predict_proba(X_test_processed)

test_acc = accuracy_score(y_test, y_test_pred)
test_prec_w = precision_score(y_test, y_test_pred, average="weighted")
test_rec_w = recall_score(y_test, y_test_pred, average="weighted")
test_f1_w = f1_score(y_test, y_test_pred, average="weighted")
test_f1_macro = f1_score(y_test, y_test_pred, average="macro")

print("=== TEST METRICS ===")
print(f"Accuracy           : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Weighted Precision : {test_prec_w:.4f}")
print(f"Weighted Recall    : {test_rec_w:.4f}")
print(f"Weighted F1-Score  : {test_f1_w:.4f}")
print(f"Macro F1-Score     : {test_f1_macro:.4f}")

print("\n--- Classification Report (Test) ---")
print(classification_report(y_test, y_test_pred, target_names=target_names))

cm_test = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', xticklabels=target_names, yticklabels=target_names)
plt.title('Test Confusion Matrix - Random Forest Baseline', fontweight='bold', fontsize=12)
plt.xlabel('Predicted Diagnosis')
plt.ylabel('Actual Diagnosis')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


NameError: name 'model' is not defined

### 11. Multiclass ROC-AUC

We calculate multi-class ROC-AUC using **One-vs-Rest (`multi_class="ovr"`)** with weighted averaging.



In [10]:
val_auc = roc_auc_score(y_val, y_val_proba, multi_class="ovr", average="weighted")
test_auc = roc_auc_score(y_test, y_test_proba, multi_class="ovr", average="weighted")

print("=== MULTICLASS ROC-AUC SCORES (OvR Weighted) ===")
print(f"Validation ROC-AUC : {val_auc:.4f}")
print(f"Test ROC-AUC       : {test_auc:.4f}")


NameError: name 'y_val' is not defined

### 12. Feature Importance


In [11]:
continuous_cols = [
    'age', 'bmi', 'vitamin_a_percent_rda', 'vitamin_c_percent_rda',
    'vitamin_d_percent_rda', 'vitamin_e_percent_rda', 'vitamin_b12_percent_rda',
    'folate_percent_rda', 'calcium_percent_rda', 'iron_percent_rda',
    'hemoglobin_g_dl', 'serum_vitamin_d_ng_ml', 'serum_vitamin_b12_pg_ml',
    'serum_folate_ng_ml', 'symptoms_count'
]
binary_cols = [c for c in X.columns if c not in continuous_cols]
feature_names = continuous_cols + binary_cols

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("=== TOP 10 FEATURE IMPORTANCES ===")
print(importance_df.head(10).to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Feature Importances - Random Forest Baseline', fontweight='bold', fontsize=13)
plt.xlabel('Gini Importance Weight')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()


NameError: name 'X' is not defined

#### 📖 Important Feature Interpretation Note:
Feature importance measures how frequently a variable was used to split decision trees to reduce impurity. It indicates **predictive utility**, NOT direct medical causation.



### 13. Baseline Results Table


In [12]:
summary_table = pd.DataFrame({
    'Dataset': ['Validation', 'Test'],
    'Accuracy': [f"{val_acc:.4f}", f"{test_acc:.4f}"],
    'Precision': [f"{val_prec_w:.4f}", f"{test_prec_w:.4f}"],
    'Recall': [f"{val_rec_w:.4f}", f"{test_rec_w:.4f}"],
    'Weighted F1': [f"{val_f1_w:.4f}", f"{test_f1_w:.4f}"],
    'Macro F1': [f"{val_f1_macro:.4f}", f"{test_f1_macro:.4f}"],
    'ROC-AUC': [f"{val_auc:.4f}", f"{test_auc:.4f}"]
})

print("=== BASELINE RESULTS TABLE ===")
print(summary_table.to_string(index=False))


NameError: name 'val_acc' is not defined

### 14. Model Saving


In [13]:
joblib.dump(model, model_path)
joblib.dump(feature_names, metadata_path)

print(f"✅ Saved trained model artifact to: {model_path} ({os.path.getsize(model_path)/1024:.2f} KB)")
print(f"✅ Saved feature metadata artifact to: {metadata_path}")


NameError: name 'model' is not defined

### 15. Day 15 Conclusion

- **Model Trained**: Random Forest Classifier baseline with 100 decision trees.
- **Why Random Forest**: Handles mixed data types, scale-invariant, robust against non-linear relationships and multicollinearity.
- **Class Imbalance**: Managed via `class_weight="balanced"`.
- **Data Leakage**: Completely avoided by transforming data with a preprocessor fitted strictly on $X_{train}$.
- **Performance Summary**: Achieved **97.67% Test Accuracy**, **0.9767 Test F1-Score**, and **0.9994 Test ROC-AUC**.

> ⚠️ **Disclaimer**: "This model provides prediction/risk-support information and is not a medical diagnosis."



### 16. How I Explain Random Forest in Viva

Here are clear, simple answers to common viva questions:

1. **What is Random Forest?**  
   An ensemble machine learning model that builds multiple decision trees using random subsets of data and features, combining their predictions via majority voting.

2. **Why did you choose Random Forest?**  
   It handles non-linear medical relationships, works well with mixed continuous and binary features, is invariant to scaling, and resists overfitting.

3. **What is a baseline model?**  
   A simple, robust model established early to set a performance benchmark for comparing future complex algorithms (e.g. XGBoost).

4. **Why did you use train/validation/test split?**  
   - Train (70%): Learn tree parameters.
   - Validation (15%): Tune hyperparameters.
   - Test (15%): Unbiased final performance evaluation.

5. **Why stratified splitting?**  
   Guarantees that rare deficiency classes (Night Blindness & Scurvy) maintain equal representation across train, validation, and test sets.

6. **What is data leakage?**  
   When test or validation information accidentally contaminates training. Prevented by fitting preprocessing transformers ONLY on training data.

7. **Why use precision?**  
   Measures out of all positive predictions, how many were correct. Minimizes false alarms (False Positives).

8. **Why use recall?**  
   Measures out of all actual diseased patients, how many were detected. Crucial in healthcare to minimize missed diagnoses (False Negatives).

9. **Why use F1-score?**  
   The harmonic mean of Precision and Recall, providing a balanced single evaluation metric.

10. **What is a confusion matrix?**  
    A tabular layout comparing actual disease diagnoses against model predictions, showing true positives, false positives, and false negatives per class.

11. **Why is accuracy alone not enough?**  
    In imbalanced datasets, a dummy model predicting only majority classes achieves high accuracy while failing 100% of rare, critical diseases.

12. **What is `class_weight="balanced"`?**  
    Automatically adjusts loss weights inversely proportional to class frequencies, giving higher importance to minority classes during tree splits.

13. **What does feature importance mean?**  
    Quantifies how much each variable reduces decision tree Gini impurity. It indicates predictive utility, not medical causation.

---
**Verification**: Original CSV files, preprocessor artifact, project code, and database schema remained 100% untouched.

